## Pandas natural lang query

In [ ]:
# 15th August, 2026
# Natural language query on  pandas Dataframe
# Ref: https://developers.llamaindex.ai/python/examples/query_engine/pandas_query_engine/
# CSV files: https://github.com/datablist/sample-csv-files

In [1]:
%reset -f

In [2]:
#1.0 Call libraries
#    Requires: 
#      pip install llama-index llama-index-experimental pandas

from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama
from llama_index.core import Settings

# File/dir readers
from llama_index.core import SimpleDirectoryReader

# PandasQueryEngine is a tool within the LlamaIndex framework 
# that lets you query a Pandas DataFrame using natural language
#from llama_index.experimental.query_engine import PandasQueryEngine
from llama_index.experimental.query_engine.pandas import PandasQueryEngine

# Global variables
from llama_index.core import Settings
import pandas as pd

/tmp/ipykernel_59240/727900349.py:15: DeprecationWarning: llama-index-experimental is deprecated and no longer maintained. It will not receive any further updates.
  from llama_index.experimental.query_engine.pandas import PandasQueryEngine


In [3]:
# 1.0.1 Initialise Settings also
Settings.llm = None
Settings.embed_model = None

/home/ashok/langchain/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LLM is explicitly disabled. Using MockLLM.
Embeddings have been explicitly disabled. Using MockEmbedding.


In [4]:

# 1.0.2
# Ollama cloud:
#   Requires 'ollama sigin' on ubuntu. AND 'Connect Device'
#   Then go to the browser, siginin and get API key.
#   api_key is NOT required if you are using 'ollama sigin'

import os
OLLAMA_API_KEY="NOT REQUIRED"
api_key = os.getenv("OLLAMA_API_KEY")


In [ ]:
# 1.1 Initialise llms
#     lightweight Cloud models that work well for this task are: 
#        "minimax-m3:cloud" , # "gpt-oss:20b-cloud" , #  "nemotron-3-nano:30b-cloud",
#         "gemma4:cloud" 
# No need to execute 'ollama run command'


llm = Ollama(model="granite4.1:3b",     # "minimax-m3:cloud"
             request_timeout=120.0,
             temperature = 0.9
            )

embed_model = OllamaEmbedding(
                                model_name="nomic-embed-text",      # Using foundational model may be overkill
                                base_url="http://localhost:11434",
                              )


Settings.llm = llm
Settings.embed_model = embed_model

In [6]:
# 1.02 Check if the API key is set correctly
#      ANd cloud model is available
llm.complete("What is the capital of France?")

INFO:httpx:HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


CompletionResponse(text='The capital of France is Paris.', additional_kwargs={}, raw={'model': 'granite4.1:3b', 'created_at': '2026-08-18T09:10:30.811997929Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11819708235, 'load_duration': 10554689822, 'prompt_eval_count': 15, 'prompt_eval_duration': 307241195, 'eval_count': 8, 'eval_duration': 905016889, 'message': Message(role='assistant', content='The capital of France is Paris.', thinking=None, images=None, tool_name=None, tool_calls=None), 'logprobs': None, 'usage': {'prompt_tokens': 15, 'completion_tokens': 8, 'total_tokens': 23}}, logprobs=None, delta=None)

In [7]:
# 1.2 Read csv file
df = pd.read_csv("/home/ashok/lprojects/customers-100.csv")

# 1.3
df.columns
df.head()
df.shape

(100, 12)

In [8]:
# 2. Initialize the query engine
# Setting verbose=True lets you see the generated pandas code
query_engine = PandasQueryEngine(df=df, 
                                 llm = llm, 
                                 verbose=True,             # see the generated pandas code
                                 synthesize_response=True  # Synthesizes Response: Puts the output in natural language
                                )



In [9]:
# 3. Query the data in natural language   Aaronstad
# response = query_engine.query("Give me the 'FirstName' of all those who are in City of 'East Leonard'")
response = query_engine.query("Give me the 'FirstName' of all those who are in City of 'Isabelborough'")
print(response)

INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
> Pandas Instructions:
```
df.loc[df['City'] == 'Isabelborough', 'FirstName']
```
> Pandas Output: 2    Roy
Name: FirstName, dtype: object
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
The first names of individuals located in the City of Isabelborough are:

Roy


In [ ]:
###################